<div style="direction:rtl;text-align:right;font-family:B Lotus, B Nazanin, Tahoma">

# تست مدل‌های تشخیص المان روی برد (بدون آموزش)

سه مدل، پشت سر هم:
1. **`any`**: فقط *جای* هر المان (یک رنگ)
2. **`fine`**: نوع هر المان (مقاومت، خازن، IC ...)؛ با `any` ترکیب می‌شود
3. **درخت تصمیم (جنگل تصادفی)**: در بلوک ۲ نوع همه‌ی المان‌ها را می‌گوید؛ در بلوک ۳ برای المان‌هایی که مدل دوم نام نداد یا اطمینانش کمتر از ۳۰٪ بود

| بلوک | کار | سرعت |
|---|---|---|
| ⚙️ تنظیمات | آستانه‌ها و روش ادغام (اسلایدر) | فوری |
| 1 آماده‌سازی | Drive، کتابخانه، بارگذاری ۳ مدل، بردهای تست | یک بار |
| 2 انتخاب عکس | شماره‌ی برد تست، تصادفی، یا عکس خودت | فوری |
| 3 | بلوک ۱: فقط مدل اول (جای المان‌ها) + نقشه‌ی خطا | فوری (از کش) |
| 4 | بلوک ۲: مدل اول + درخت تصمیم برای همه + نقشه‌ی خطا | بار اول کندتر |
| 5 ✅ | بلوک ۳ (اصلی): همه‌ی المان‌ها، مدل دوم نام می‌دهد، درخت برای بی‌نام‌ها و نام‌های ضعیف | بار اول کمی کندتر |

**اجرای اول:** `Runtime ▸ Run all`. بعد از آن فقط بلوک‌هایی را که لازم داری اجرا کن.

**تنظیم سریع:** هر بلوک نمایش (۳ تا ۵) اسلایدر «حداقل اطمینان» و «حالت» خودش را دارد: عوض کن و همان بلوک را دوباره اجرا کن. اگر عکس و حالت عوض نشده باشد، مدل‌ها دوباره اجرا نمی‌شوند (نتیجه‌ی خام نگه داشته شده) و جواب فوری است. بقیه‌ی تنظیم‌ها در بلوک ⚙️ است.

**GPU:** `Runtime ▸ Change runtime type ▸ T4 GPU`. اگر GPU نبود خودکار روی CPU اجرا می‌شود (کندتر؛ زمان‌ها همان را نشان می‌دهند).

**درصد دقت:** روی بردهای تست (برچسب دارند) دقت واقعی نشان داده می‌شود: precision (چند درصد جعبه‌ها درست‌اند)، recall (چند درصد المان‌ها پیدا شدند)، F1، درصد نوع درست، و تعداد تکراری‌ها. روی عکس خودت بدون برچسب فقط میانگین اطمینان مدل نشان داده می‌شود.

فایل‌های لازم، همه در پوشه‌ی `runs_hr` روی Drive:
`final/any_best.pt`، `final/fine_best.pt`، پوشه‌ی `tree/` (مدل درخت)، `pcb_data_hr.zip` (فقط برای بردهای تست)

</div>

In [ ]:
#@title ⚙️ تنظیمات — مقدارها را عوض کن، این بلوک را اجرا کن، بعد یکی از بلوک‌های نمایش (۳ تا ۵) را
#@markdown ### ۱) آستانه‌ها و ادغام جعبه‌ها (فوری؛ مدل دوباره اجرا نمی‌شود)
#@markdown حداقل اطمینان برای قبول یک المان (کمتر = المان بیشتر، و خطای بیشتر)
حداقل_اطمینان_پیدا_کردن = 0.15  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
حداقل_اطمینان_نوع = 0.15  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
#@markdown اگر همپوشانی دو جعبه‌ی هم‌نوع بیشتر از این باشد، یکی حذف می‌شود
همپوشانی_حذف_تکراری = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
#@markdown «هوشمند» = رفع چند بار دیده شدن المان‌های بزرگ (پیشنهاد) | «ساده» = روش قبلی
روش_ادغام = "هوشمند"  #@param ["هوشمند", "ساده"]
#@markdown تکه‌ای از المان بزرگ که لبه‌ی برش عکس آن را بریده، اگر این مقدار از آن داخل یک المان کامل باشد حذف می‌شود
سهم_داخل_المان_بزرگ = 0.6  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown تکه‌های یک المان که در دو برش کنار هم افتاده‌اند یکی شوند
یکی_کردن_تکه_ها = True  #@param {type:"boolean"}
#@markdown جعبه‌ی تکراری هم‌اندازه: اگر این مقدار داخل جعبه‌ی دیگر باشد **و** اندازه‌اش نزدیک آن باشد حذف می‌شود (المان ریز زیر المان بزرگ حذف نمی‌شود)
سهم_داخل_تکراری = 0.85  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
حداقل_نسبت_اندازه_تکراری = 0.35  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
#@markdown ### ۲) ترکیب سه مدل
#@markdown همپوشانی لازم برای اینکه نوعِ مدل دوم به جعبه‌ی مدل اول داده شود
همپوشانی_تطبیق_دو_مدل = 0.4  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
#@markdown المانی که فقط مدل دوم پیدا کرده، با این اطمینان یا بیشتر اضافه می‌شود
اطمینان_افزودن_از_مدل_دوم = 0.5  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
#@markdown درخت تصمیم فقط وقتی نوع را تعیین می‌کند که احتمالش حداقل این باشد
حداقل_اطمینان_درخت = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown اگر بیشتر از صفر باشد، درخت برای نوع‌هایی که مدل دوم با اطمینان کمتر از این داده هم نظر می‌دهد
درخت_برای_نوع_ضعیف = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown المان‌های کوچک‌تر از این (پیکسل) به درخت داده نمی‌شوند
کمترین_اندازه_برای_درخت = 12  #@param {type:"slider", min:4, max:64, step:2}
#@markdown ### ۳) نمایش
نوشته_روی_جعبه = "خودکار"  #@param ["خودکار", "نام و درصد", "فقط درصد", "هیچ"]
نمایش_نقشه_خطا = True  #@param {type:"boolean"}
#@markdown ### ۴) تشخیص (تغییر این‌ها مدل‌ها را دوباره اجرا می‌کند؛ بعد از تغییر «پردازنده» یا «مدل درخت» بلوک ۱ را هم دوباره اجرا کن)
پردازنده = "خودکار"  #@param ["خودکار", "GPU", "CPU"]
اندازه_برش = 1024  #@param {type:"integer"}
همپوشانی_برش = 0.25  #@param {type:"slider", min:0.0, max:0.5, step:0.05}
اندازه_ورودی_مدل = 1024  #@param {type:"integer"}
#@markdown عکس‌هایی که المان‌هایشان بزرگ است مثل داده‌ی آموزش کوچک می‌شوند (خاموش = «مقیاس_دستی»)
مقیاس_خودکار = True  #@param {type:"boolean"}
مقیاس_دستی = 1.0  #@param {type:"number"}
#@markdown یک بار هم کل برد یکجا دیده شود (برای المان‌های بزرگ)
نگاه_به_کل_برد = True  #@param {type:"boolean"}
مدل_درخت = "بردهای واقعی"  #@param ["بردهای واقعی", "مدل اصلی"]
#@markdown ### ۵) پوشه در Google Drive
شناسه_پوشه = "1yq4gdsJe_sAKt0c10tzWhBHi40pEiiop"  #@param {type:"string"}
نام_پوشه = "runs_hr"  #@param {type:"string"}
مسیر_دستی_پوشه = ""  #@param {type:"string"}

# ---- تبدیل به نام‌هایی که بقیه‌ی نوت‌بوک استفاده می‌کند (لازم نیست تغییر کند) ----
CONF_ANY, CONF_FINE, IOU = حداقل_اطمینان_پیدا_کردن, حداقل_اطمینان_نوع, همپوشانی_حذف_تکراری
MERGE_MODE = {"هوشمند": "smart", "ساده": "simple"}[روش_ادغام]
LABEL_TEXT = {"خودکار": "auto", "نام و درصد": "names", "فقط درصد": "conf", "هیچ": "none"}[نوشته_روی_جعبه]
SHOW_ERRORS = نمایش_نقشه_خطا
TILE, OVERLAP, IMGSZ = int(اندازه_برش), float(همپوشانی_برش), int(اندازه_ورودی_مدل)
TREE_MODEL = {"بردهای واقعی": "boards", "مدل اصلی": "original"}[مدل_درخت]
RUNS_FOLDER_ID, RUNS_NAME, RUNS_DIR = شناسه_پوشه, نام_پوشه, مسیر_دستی_پوشه

import os, warnings
warnings.filterwarnings("ignore")
os.environ["YOLO_VERBOSE"] = "False"
import torch
os.makedirs("/content/work", exist_ok=True)
_GPU = torch.cuda.is_available()
if پردازنده == "GPU" and not _GPU:
    print("⚠️ GPU در دسترس نیست، روی CPU اجرا می‌شود (Runtime ▸ Change runtime type ▸ T4 GPU)")
DEV = 0 if (_GPU and پردازنده != "CPU") else "cpu"
CFG = dict(tile=TILE, overlap=OVERLAP, imgsz=IMGSZ, global_pass=bool(نگاه_به_کل_برد), edge_px=4,
           half=DEV != "cpu", max_det=1000, batch=8 if DEV != "cpu" else 2, device=DEV,
           auto_scale=bool(مقیاس_خودکار), auto_scale_tol=1.2, scale=float(مقیاس_دستی))
P = dict(mode=MERGE_MODE, conf_any=CONF_ANY, conf_fine=CONF_FINE, iou=IOU, cross_iou=0.7, contain=سهم_داخل_المان_بزرگ,
         join=یکی_کردن_تکه_ها, join_align=0.6, dup_ios=سهم_داخل_تکراری, dup_area=حداقل_نسبت_اندازه_تکراری,
         global_min_px=0.25 * TILE, match_iou=همپوشانی_تطبیق_دو_مدل, add_fine_conf=اطمینان_افزودن_از_مدل_دوم,
         tree_min_prob=حداقل_اطمینان_درخت, tree_if_fine_below=درخت_برای_نوع_ضعیف, tree_min_px=کمترین_اندازه_برای_درخت,
         tree_weight=0.4, tree_skip_above=0.85, fine_min=0.3)
print("اجرا روی:", "GPU" if DEV != "cpu" else "CPU", "| روش ادغام:", روش_ادغام,
      "| حداقل اطمینان:", CONF_ANY, "/", CONF_FINE, "| مدل درخت:", مدل_درخت)

<div style="direction:rtl;text-align:right;font-family:B Lotus, B Nazanin, Tahoma">

## کد (دو فایل؛ لازم نیست بازشان کنی - فقط اجرا شوند)

</div>

In [ ]:
%%writefile /content/work/pcb_infer.py
"""Test-only engine of the PCB detector: no training here.

  detect      tiled YOLO pass (+ one whole-board pass for big parts), raw results CACHED at a low confidence,
              so changing CONF / IOU / merge settings re-filters in milliseconds without running the model again
  postprocess the fix for big parts detected several times (see merge_big_parts)
  fuse        'any' (where) + 'fine' (what) + random forest (for the parts YOLO could not name)
  evaluate    precision / recall / F1 against the board's labels (dataset boards), duplicates counted
  draw        1) elements in one colour  2) colour per type  3) error map (correct / missed / false / duplicate)
"""
import colorsys
import hashlib
import os
import time
import warnings

warnings.filterwarnings("ignore")          # numpy / sklearn / torch notices would repeat for every part
os.environ.setdefault("YOLO_VERBOSE", "False")

import cv2
import numpy as np
import torch
from torchvision.ops import batched_nms, box_iou

RAW_CONF = 0.05          # raw detections are kept down to this score; CONF can then be changed freely above it
TRAIN_MAX_PART = 64      # training photos were shrunk until their median part was at most this many px


# ================================================================== timing
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


class Clock:
    """with Clock() as c: ...  -> c.ms"""
    def __enter__(self):
        _sync(); self.t = time.perf_counter(); return self

    def __exit__(self, *a):
        _sync(); self.ms = (time.perf_counter() - self.t) * 1000


def load_yolo(path, device, imgsz=1024, half=False):
    from ultralytics import YOLO
    m = YOLO(path)
    with Clock() as c:   # warm-up: the first call builds CUDA kernels and would spoil the timing
        m.predict(np.zeros((imgsz, imgsz, 3), np.uint8), imgsz=imgsz, device=device, half=half, verbose=False)
    m.warmup_ms = c.ms
    return m


# ================================================================== raw detection (slow part, cached)
def tiles_of(length, tile, step):
    """Same tiling as the training set (tools/build_hr_dataset.py)."""
    pos = list(range(0, max(length - tile, 0) + 1, step))
    if pos[-1] + tile < length:
        pos.append(length - tile)
    return pos


def _predict(model, imgs, cfg):
    out = []
    for i in range(0, len(imgs), cfg["batch"]):
        out += model.predict(imgs[i:i + cfg["batch"]], conf=RAW_CONF, imgsz=cfg["imgsz"], device=cfg["device"],
                             half=cfg["half"], max_det=cfg["max_det"], verbose=False)
    return out


def raw_pass(model, img, cfg):
    """Every tile + (big photos) the whole board once. Boxes in the pixels of `img`.
    cut[i] = the box touches an INNER tile border, i.e. it is probably only a piece of a bigger part."""
    H, W = img.shape[:2]
    tile, edge = cfg["tile"], cfg["edge_px"]
    step = max(1, int(tile * (1 - cfg["overlap"])))
    boxes, scores, cls, cut, src = [], [], [], [], []
    if tile and max(H, W) > tile:
        pos = [(x, y) for y in tiles_of(H, tile, step) for x in tiles_of(W, tile, step)]
        crops = [img[y:y + tile, x:x + tile] for x, y in pos]
        for (x, y), r in zip(pos, _predict(model, crops, cfg)):
            b = r.boxes.xyxy.cpu().numpy().copy()
            tw, th = min(tile, W - x), min(tile, H - y)
            c = np.zeros(len(b), bool)
            if x > 0:          c |= b[:, 0] <= edge
            if y > 0:          c |= b[:, 1] <= edge
            if x + tw < W:     c |= b[:, 2] >= tw - edge
            if y + th < H:     c |= b[:, 3] >= th - edge
            b[:, [0, 2]] += x; b[:, [1, 3]] += y
            boxes.append(b); scores.append(r.boxes.conf.cpu().numpy()); cls.append(r.boxes.cls.cpu().numpy().astype(int))
            cut.append(c); src.append(np.zeros(len(b), int))
        n_tiles = len(pos)
        if cfg["global_pass"]:   # the whole board in one look: big parts are seen complete here
            r = _predict(model, [img], cfg)[0]
            b = r.boxes.xyxy.cpu().numpy()
            boxes.append(b); scores.append(r.boxes.conf.cpu().numpy()); cls.append(r.boxes.cls.cpu().numpy().astype(int))
            cut.append(np.zeros(len(b), bool)); src.append(np.ones(len(b), int))
    else:
        r = _predict(model, [img], cfg)[0]
        boxes.append(r.boxes.xyxy.cpu().numpy()); scores.append(r.boxes.conf.cpu().numpy())
        cls.append(r.boxes.cls.cpu().numpy().astype(int)); cut.append(np.zeros(len(boxes[-1]), bool))
        src.append(np.zeros(len(boxes[-1]), int))
        n_tiles = 1
    cat = lambda v, d, s: np.concatenate(v) if v else np.zeros(s, d)
    return dict(boxes=cat(boxes, float, (0, 4)).reshape(-1, 4), scores=cat(scores, float, 0), cls=cat(cls, int, 0),
                cut=cat(cut, bool, 0), src=cat(src, int, 0), n_tiles=n_tiles, size=(W, H))


def _resize(img, s):
    return img if s == 1 else cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)


_CACHE = {}


def image_key(img):
    return hashlib.md5(img[::7, ::7].tobytes()).hexdigest() + f"{img.shape}"


def detect_raw(model, img, cfg, scale=None, tag=""):
    """Raw detections of one model, in ORIGINAL photo pixels. Cached per (model, photo, scale, tile settings).
    scale None = automatic: if the parts look bigger than in training, the photo is shrunk like in training."""
    keys = ("tile", "overlap", "imgsz", "global_pass", "edge_px", "half", "max_det")
    base = (tag or id(model), image_key(img)) + tuple(cfg[k] for k in keys)
    probe_ms = 0.0
    if scale is None:
        if cfg["auto_scale"]:
            r1 = detect_raw(model, img, cfg, 1.0, tag)
            sel = r1["scores"] >= 0.25
            b = r1["boxes"][sel]
            med = float(np.median(np.minimum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]))) if len(b) >= 3 else 0.0
            scale = 1.0
            if med > TRAIN_MAX_PART * cfg["auto_scale_tol"]:
                scale = round(TRAIN_MAX_PART / med, 3)
                probe_ms = r1["ms"]
            if scale == 1.0:
                return dict(r1, probe_ms=0.0, median_part=med)
        else:
            scale = cfg["scale"]
    key = base + (scale,)
    if key not in _CACHE:
        with Clock() as c:
            r = raw_pass(model, _resize(img, scale), cfg)
        r["boxes"] = r["boxes"] / scale
        r.update(scale=scale, ms=c.ms, cached=False)
        _CACHE[key] = r
    else:
        _CACHE[key]["cached"] = True
    return dict(_CACHE[key], probe_ms=probe_ms)


def clear_cache():
    _CACHE.clear()


# ================================================================== post-processing (fast)
def _area(b):
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)


def _inter(a, b):
    """Intersection area matrix [len(a), len(b)]."""
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    return np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)


def merge_big_parts(B, S, C, cut, src, p, agnostic):
    """The fix for 'one big part -> several boxes', without deleting small parts that sit under a big one.

    Why it happens: a part bigger than the tile overlap is seen only in pieces, and every tile reports its piece
    (the training labels also kept pieces >= 50 % visible). Plain NMS cannot remove them: a piece covers only a
    part of the full box, so their IoU is low (and pieces of neighbouring tiles may not overlap at all).

    1  pieces inside a bigger COMPLETE box (from a tile where it fits, or from the whole-board pass) are dropped
       -> only boxes that touch an inner tile border are candidates, so a small part lying under a big one,
          seen whole in its tile, is never removed
    2  remaining pieces of the same part in neighbouring tiles are joined into one box (union)
    3  near-same-size duplicates (one box almost inside another, similar area) are removed; a small part under
       a big one has a much smaller area, so it is kept
    """
    keep = np.ones(len(B), bool)
    A = _area(B)
    if len(B) == 0:
        return B, S, C, np.zeros(0, int)
    # global-pass boxes only bring the big parts
    small_global = (src == 1) & (np.maximum(B[:, 2] - B[:, 0], B[:, 3] - B[:, 1]) < p["global_min_px"])
    keep &= ~small_global
    I = _inter(B, B)
    ios = I / np.maximum(A[:, None], 1e-6)            # ios[i, j] = share of box i inside box j
    same = np.ones_like(I, bool) if agnostic else (C[:, None] == C[None, :])
    # 1) piece inside a bigger complete box
    if p["contain"] > 0:
        whole = keep & ~cut
        cond = (ios >= p["contain"]) & whole[None, :] & (A[None, :] > 1.2 * A[:, None])
        keep &= ~(cut & cond.any(1))
    # 2) join pieces across tile borders
    group = np.arange(len(B))
    if p["join"]:
        idx = np.where(keep & cut)[0]
        def root(i):
            while group[i] != i:
                group[i] = group[group[i]]; i = group[i]
            return i
        for a_i, i in enumerate(idx):
            for j in idx[a_i + 1:]:
                if not same[i, j] or I[i, j] <= 0:
                    continue
                # pieces of one part line up: they share most of their extent across the border
                ox = min(B[i, 2], B[j, 2]) - max(B[i, 0], B[j, 0]); oy = min(B[i, 3], B[j, 3]) - max(B[i, 1], B[j, 1])
                ax = ox / max(min(B[i, 2] - B[i, 0], B[j, 2] - B[j, 0]), 1e-6)
                ay = oy / max(min(B[i, 3] - B[i, 1], B[j, 3] - B[j, 1]), 1e-6)
                if max(ax, ay) >= p["join_align"] and min(ax, ay) > 0:
                    group[root(i)] = root(j)
        roots = np.array([root(i) for i in range(len(B))]) if len(idx) else group
        for r in np.unique(roots[idx]) if len(idx) else []:
            members = idx[roots[idx] == r]
            if len(members) > 1:
                best = members[np.argmax(S[members])]
                B[best] = [B[members, 0].min(), B[members, 1].min(), B[members, 2].max(), B[members, 3].max()]
                keep[members] = False; keep[best] = True
        A = _area(B); I = _inter(B, B); ios = I / np.maximum(A[:, None], 1e-6)
    n_merged = np.zeros(len(B), int)
    # 3) same-size duplicates
    if p["dup_ios"] > 0:
        order = np.argsort(-S)
        for k, i in enumerate(order):
            if not keep[i]:
                continue
            for j in order[k + 1:]:
                if keep[j] and same[i, j] and ios[j, i] >= p["dup_ios"] and A[j] >= p["dup_area"] * A[i] and A[j] <= A[i] / max(p["dup_area"], 1e-6):
                    keep[j] = False; n_merged[i] += 1
    return B[keep], S[keep], C[keep], n_merged[keep]


def postprocess(raw, conf, p, agnostic):
    """raw -> final boxes (original pixels). p['mode']: 'smart' (big-part fix) or 'simple' (NMS only, as before)."""
    sel = raw["scores"] >= conf
    B, S, C = raw["boxes"][sel].copy(), raw["scores"][sel], raw["cls"][sel]
    cut, src = raw["cut"][sel], raw["src"][sel]
    if agnostic:
        C = np.zeros_like(C)
    if p["mode"] == "simple":
        keep = (src == 0)
        B, S, C = B[keep], S[keep], C[keep]
    else:
        B, S, C, _ = merge_big_parts(B, S, C, cut, src, p, agnostic)
    if len(B):
        tB, tS, tC = torch.tensor(B, dtype=torch.float32), torch.tensor(S, dtype=torch.float32), torch.tensor(C)
        k = batched_nms(tB, tS, tC, p["iou"]).numpy()
        B, S, C = B[k], S[k], C[k]
        if not agnostic and p["cross_iou"] < 1:      # one part, two names: keep the more confident one
            k = batched_nms(torch.tensor(B, dtype=torch.float32), torch.tensor(S, dtype=torch.float32),
                            torch.zeros(len(B), dtype=torch.long), p["cross_iou"]).numpy()
            B, S, C = B[k], S[k], C[k]
    o = np.argsort(-S)
    return B[o], S[o], C[o]


# ================================================================== fusion of the three models
def fuse(any_det, fine_det, fine_names, p):
    """Each 'any' box gets the name of the best-overlapping 'fine' box; unmatched -> '?'.
    Confident fine boxes that 'any' missed are added. Returns dict of lists."""
    Ba, Sa = any_det[0], any_det[1]
    Bf, Sf, Cf = fine_det
    labels, lconf, srcs = ["?"] * len(Ba), [0.0] * len(Ba), ["any"] * len(Ba)
    used = np.zeros(len(Bf), bool)
    if len(Ba) and len(Bf):
        iou = box_iou(torch.tensor(Ba, dtype=torch.float32), torch.tensor(Bf, dtype=torch.float32)).numpy()
        for i, j in sorted(zip(*np.where(iou >= p["match_iou"])), key=lambda ij: -iou[ij]):
            if labels[i] == "?" and not used[j]:
                labels[i], lconf[i], srcs[i] = fine_names[int(Cf[j])], float(Sf[j]), "fine"; used[j] = True
    B, S = list(Ba), list(Sa)
    for j in np.where(~used & (Sf >= p["add_fine_conf"]))[0]:
        if len(Ba) and box_iou(torch.tensor(Bf[j:j + 1], dtype=torch.float32), torch.tensor(Ba, dtype=torch.float32)).max() >= 0.3:
            continue
        B.append(Bf[j]); S.append(Sf[j]); labels.append(fine_names[int(Cf[j])]); lconf.append(float(Sf[j])); srcs.append("fine+")
    return dict(boxes=np.array(B).reshape(-1, 4), scores=np.array(S), labels=labels, label_conf=lconf, src=srcs)


def crop(img, b, pad=0.08):
    H, W = img.shape[:2]
    x0, y0, x1, y1 = b
    px, py = (x1 - x0) * pad, (y1 - y0) * pad
    return img[int(max(y0 - py, 0)):int(min(y1 + py, H)), int(max(x0 - px, 0)):int(min(x1 + px, W))]


def tree_refine(res, img, rf, p, to_name):
    """Random forest for the boxes YOLO could not name ('?') and, optionally, for weak YOLO names.
    The forest only overrides when it is confident (prob >= tree_min_prob)."""
    import tree_features as tf
    idx = [i for i, (l, c) in enumerate(zip(res["labels"], res["label_conf"]))
           if l == "?" or (p["tree_if_fine_below"] > 0 and c < p["tree_if_fine_below"])]
    idx = [i for i in idx if min(res["boxes"][i][2] - res["boxes"][i][0], res["boxes"][i][3] - res["boxes"][i][1]) >= p["tree_min_px"]]
    with Clock() as c:
        out = tf.classify(rf, [crop(img, res["boxes"][i]) for i in idx]) if idx else []
    changed = 0
    for i, (cl, pr) in zip(idx, out):
        if pr >= p["tree_min_prob"] and pr > res["label_conf"][i]:
            res["labels"][i], res["label_conf"][i], res["src"][i] = to_name(cl), pr, "tree"; changed += 1
    return dict(n_asked=len(idx), n_changed=changed, ms=c.ms)


# ================================================================== accuracy against labels
def read_labels(txt, W, H, names, drop=()):
    B, L = [], []
    if txt and os.path.exists(txt):
        for l in open(txt, encoding="utf8"):
            v = l.split()
            if len(v) == 5 and names[int(v[0])] not in drop:
                c, x, y, w, h = int(v[0]), *map(float, v[1:])
                B.append([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]); L.append(names[c])
    return np.array(B).reshape(-1, 4), L


def evaluate(B, S, labels, GB, GL, iou_thr=0.5, typed=True):
    """Greedy matching by score at IoU >= iou_thr. Returns detection P/R/F1, typed P/R/F1, duplicates and
    per-box status (for the error map)."""
    n, g = len(B), len(GB)
    st_pred, st_gt = ["fp"] * n, ["miss"] * g
    tp = tp_typed = 0
    if n and g:
        iou = box_iou(torch.tensor(B, dtype=torch.float32), torch.tensor(GB, dtype=torch.float32)).numpy()
        I = _inter(np.asarray(B, float), np.asarray(GB, float)); ios = I / np.maximum(_area(np.asarray(B, float))[:, None], 1e-6)
        taken = np.zeros(g, bool)
        for i in np.argsort(-np.asarray(S)):
            cand = np.where(~taken & (iou[i] >= iou_thr))[0]
            if len(cand):
                j = cand[np.argmax(iou[i, cand])]
                taken[j] = True; tp += 1
                ok = (not typed) or labels[i] == GL[j]
                tp_typed += ok
                st_pred[i] = "ok" if ok else "wrong_type"; st_gt[j] = "found"
        for i in range(n):
            if st_pred[i] != "fp":
                continue
            if ((ios[i] >= 0.6) & taken).any():      # lies on a part that was already found = duplicate piece
                st_pred[i] = "dup"
            elif iou[i].max() >= 0.1:                 # touches a label but IoU < threshold = box drawn too big/small/shifted
                st_pred[i] = "loc"
            # else stays "fp": no label anywhere near. On these boards many such boxes are REAL parts that the
            # people who labelled the board skipped (check them on the error map), so precision is a lower bound
    f1 = lambda p_, r_: 2 * p_ * r_ / max(p_ + r_, 1e-9)
    P, R = tp / max(n, 1), tp / max(g, 1)
    Pt, Rt = tp_typed / max(n, 1), tp_typed / max(g, 1)
    return dict(n_pred=n, n_gt=g, tp=tp, P=P, R=R, F1=f1(P, R), dup=st_pred.count("dup"),
                loc=st_pred.count("loc"), unl=st_pred.count("fp"),
                type_acc=tp_typed / max(tp, 1), Pt=Pt, Rt=Rt, F1t=f1(Pt, Rt), st_pred=st_pred, st_gt=st_gt)


# ================================================================== drawing (same look as predict.py)
SHORT = {"Resistor": "R", "Capacitor": "C", "Electrolytic Capacitor": "EC", "IC": "IC", "Inductor": "L", "Diode": "D",
         "LED": "LED", "Transistor": "Q", "Oscillator": "X", "Connector": "CN", "Switch/Button": "SW", "Other": "O",
         "Marker": "M", "component": "", "?": "?"}
ELEMENT_COLOUR = (0, 255, 120)
FIXED = {}


def palette(labels):
    out = {}
    for k in sorted(set(labels), key=lambda s: (s == "?", s)):
        if k not in FIXED:   # one colour per type that stays the same from picture to picture
            i = len(FIXED)
            r, g, b = colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.9, 1.0)
            FIXED[k] = (int(b * 255), int(g * 255), int(r * 255))
        out[k] = FIXED[k]
    if "?" in out:
        out["?"] = (210, 210, 210)
    return out


def _canvas(img, target):
    H0, W0 = img.shape[:2]
    up = target / max(H0, W0)
    up = max(up, 1.0) if max(H0, W0) < target else min(1.0, 4000 / max(H0, W0))
    out = cv2.resize(img, None, fx=up, fy=up, interpolation=cv2.INTER_CUBIC if up > 1 else cv2.INTER_AREA) if up != 1 else img.copy()
    return out, up


def _style(H, W):
    side = max(H, W)
    return max(2, round(side / 900)), max(0.7, side / 2600)


def _legend(rows, col, H, title, lines=()):
    ph = max(H, 48 * (len(rows) + len(lines) + 4))
    panel = np.full((ph, 470, 3), 28, np.uint8)
    cv2.putText(panel, title, (18, 44), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
    y = 90
    for t in lines:
        cv2.putText(panel, t, (18, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 230, 255), 2, cv2.LINE_AA); y += 40
    y += 20
    for k, n in rows:
        cv2.rectangle(panel, (18, y - 26), (52, y + 8), col[k], -1)
        cv2.putText(panel, f"{k}: {n}", (66, y + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (235, 235, 235), 2, cv2.LINE_AA)
        y += 46
    return panel


def _join(out, panel):
    H, W = out.shape[:2]
    ph = panel.shape[0]
    left = np.vstack([out, np.zeros((ph - H, W, 3), np.uint8)]) if ph > H else out
    return np.hstack([left, panel])


def draw_elements(img, boxes, lines=(), target=2400, fill=0.28):
    out, up = _canvas(img, target)
    H, W = out.shape[:2]
    lw, _ = _style(H, W)
    bx = (np.asarray(boxes) * up).round().astype(int).reshape(-1, 4)
    over = out.copy()
    for x0, y0, x1, y1 in bx:
        cv2.rectangle(over, (x0, y0), (x1, y1), ELEMENT_COLOUR, -1)
    out = cv2.addWeighted(over, fill, out, 1 - fill, 0)
    for x0, y0, x1, y1 in bx:
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 0, 0), lw + 2, cv2.LINE_AA)
        cv2.rectangle(out, (x0, y0), (x1, y1), ELEMENT_COLOUR, lw, cv2.LINE_AA)
    return _join(out, _legend([("element", len(bx))], {"element": ELEMENT_COLOUR}, H, "Elements", lines))


def draw_types(img, boxes, labels, scores, src=None, text="auto", min_label_px=24, lines=(), target=2400):
    """text: none | auto (short code on big boxes) | names (name + confidence %) | conf (confidence % only)
    Boxes named by the random forest get a white corner mark and count as '(tree)' in the legend."""
    out, up = _canvas(img, target)
    H, W = out.shape[:2]
    lw, fs = _style(H, W)
    col = palette(labels)
    bx = (np.asarray(boxes) * up).reshape(-1, 4)
    src = src or [""] * len(bx)
    for i in range(len(bx)):
        x0, y0, x1, y1 = [int(round(v)) for v in bx[i]]
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 0, 0), lw + 2, cv2.LINE_AA)
        cv2.rectangle(out, (x0, y0), (x1, y1), col[labels[i]], lw, cv2.LINE_AA)
        if src[i] == "tree":
            d = max(6, lw * 4)
            cv2.line(out, (x0, y0), (x0 + d, y0), (255, 255, 255), lw + 1); cv2.line(out, (x0, y0), (x0, y0 + d), (255, 255, 255), lw + 1)
    if text != "none" and len(bx):
        font, tt = cv2.FONT_HERSHEY_SIMPLEX, max(1, lw - 1)
        taken = []
        area = (bx[:, 2] - bx[:, 0]) * (bx[:, 3] - bx[:, 1])
        for i in np.argsort(-area):
            x0, y0, x1, y1 = [int(round(v)) for v in bx[i]]
            if text == "auto" and min(x1 - x0, y1 - y0) < min_label_px * up:
                continue
            s = {"names": f"{labels[i]} {scores[i] * 100:.0f}%", "conf": f"{scores[i] * 100:.0f}%"}.get(text, SHORT.get(labels[i], labels[i][:3]))
            if not s:
                continue
            (tw, th), _ = cv2.getTextSize(s, font, fs, tt)
            for tx, ty in ((x0, y0 - 4), (x0, y1 + th + 4), (x1 - tw, y0 - 4), (x1 - tw, y1 + th + 4)):
                r = (tx - 2, ty - th - 2, tx + tw + 2, ty + 4)
                if r[0] < 0 or r[1] < 0 or r[2] > W or r[3] > H:
                    continue
                if all(r[2] < q[0] or r[0] > q[2] or r[3] < q[1] or r[1] > q[3] for q in taken):
                    taken.append(r)
                    cv2.putText(out, s, (tx, ty), font, fs, (0, 0, 0), tt + 3, cv2.LINE_AA)
                    cv2.putText(out, s, (tx, ty), font, fs, col[labels[i]], tt, cv2.LINE_AA)
                    break
    counts = {k: labels.count(k) for k in col}
    tree_n = {k: sum(1 for l, s_ in zip(labels, src) if l == k and s_ == "tree") for k in col}
    rows = [(f"{k}" + (f" ({tree_n[k]} tree)" if tree_n[k] else ""), n) for k, n in sorted(counts.items(), key=lambda kv: -kv[1])]
    col2 = {f"{k}" + (f" ({tree_n[k]} tree)" if tree_n[k] else ""): c for k, c in col.items()}
    return _join(out, _legend(rows, col2, H, "Types", lines)), counts


ERR_COL = {"ok": (0, 220, 0), "wrong_type": (0, 215, 255), "loc": (255, 200, 0), "fp": (0, 140, 255),
           "dup": (255, 0, 255), "miss": (0, 0, 255)}
ERR_NAME = {"ok": "correct", "wrong_type": "found, wrong type", "loc": "box size/position off",
            "fp": "no label here (check!)", "dup": "duplicate piece", "miss": "missed (label)"}


def draw_errors(img, B, ev, GB, lines=(), target=2400):
    """Error map against the labels: green correct, yellow wrong type, orange false, magenta duplicate, red missed."""
    out, up = _canvas(img, target)
    H, W = out.shape[:2]
    lw, _ = _style(H, W)
    items = [(b, s) for b, s in zip(np.asarray(B).reshape(-1, 4), ev["st_pred"])] + \
            [(b, "miss") for b, s in zip(np.asarray(GB).reshape(-1, 4), ev["st_gt"]) if s == "miss"]
    for b, s in items:
        x0, y0, x1, y1 = (np.asarray(b) * up).round().astype(int)
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 0, 0), lw + 2, cv2.LINE_AA)
        cv2.rectangle(out, (x0, y0), (x1, y1), ERR_COL[s], lw, cv2.LINE_AA)
    cnt = {k: 0 for k in ERR_COL}
    for _, s in items:
        cnt[s] += 1
    rows = [(ERR_NAME[k], cnt[k]) for k in ERR_COL]
    return _join(out, _legend(rows, {ERR_NAME[k]: v for k, v in ERR_COL.items()}, H, "Errors vs labels", lines))


# ================================================================== one board, all stages
def model_tag(path):
    return f"{path}|{os.path.getmtime(path) if os.path.exists(path) else 0}"


_TREE_CACHE = {}
_TREE_MS = []   # measured ms per part, used to report the real forest time of cached answers


def tree_probs(M, img, boxes, p, ask=None):
    """Forest probabilities {type: prob} for each box (None = not asked: too small or ask[i] False).
    Results are cached per photo + box, so moving a slider does not run the forest again."""
    import tree_features as tf
    key0 = image_key(img) + M.get("tree_kind", "")
    out, todo = [None] * len(boxes), []
    for i, b in enumerate(boxes):
        if ask is not None and not ask[i]:
            continue
        if min(b[2] - b[0], b[3] - b[1]) < p["tree_min_px"]:
            continue
        k = (key0, tuple(int(round(v)) for v in b))
        if k in _TREE_CACHE:
            out[i] = _TREE_CACHE[k]
        else:
            todo.append((i, k))
    with Clock() as c:
        if todo:
            X = tf.features_many([crop(img, boxes[i]) for i, _ in todo])
            P = M["rf"].predict_proba(X)
            names = [M["to_name"](cl) for cl in M["rf"].classes_]
            for (i, k), pr in zip(todo, P):
                _TREE_CACHE[k] = out[i] = dict(zip(names, map(float, pr)))
    n = sum(o is not None for o in out)
    if todo:
        _TREE_MS.append(c.ms / len(todo))
    per = float(np.median(_TREE_MS)) if _TREE_MS else 0.0
    return out, dict(n_asked=n, n_new=len(todo), ms=c.ms + (n - len(todo)) * per)


def vote(fu, probs, p):
    """Type of each part from BOTH the 'fine' model and the forest:
    score(type) = (1 - w) * fine_confidence * [fine said this type] + w * forest_probability(type)
    w = p['tree_weight']. No fine name -> the forest decides alone (if its probability >= tree_min_prob)."""
    w = p["tree_weight"]
    labels, conf, src = [], [], []
    for lf, cf, pt, s0 in zip(fu["labels"], fu["label_conf"], probs, fu["src"]):
        if pt is None:
            labels.append(lf); conf.append(cf); src.append(s0)
            continue
        if lf == "?":
            c = max(pt, key=pt.get)
            ok = pt[c] >= p["tree_min_prob"]
            labels.append(c if ok else "?"); conf.append(pt[c] if ok else 0.0); src.append("tree" if ok else s0)
            continue
        score = {k: w * v for k, v in pt.items()}
        score[lf] = score.get(lf, 0.0) + (1 - w) * cf
        c = max(score, key=score.get)
        labels.append(c)
        conf.append(score[c] / ((1 - w) * cf + w) if ((1 - w) * cf + w) > 0 else 0.0)
        src.append("fine+tree" if c == lf and max(pt, key=pt.get) == lf else ("fine" if c == lf else "tree"))
    return labels, conf, src


def detect_multi(model, img, cfg, scale, tag):
    """detect_raw + (cfg['zoom'] > 1) one more pass on the photo ENLARGED by that factor, for boards whose parts
    are tiny (median part < cfg['zoom_below'] px): small parts that were missed at normal size are found there.
    Both passes are merged by the normal post-processing (duplicates removed)."""
    r = detect_raw(model, img, cfg, scale, tag)
    z = cfg.get("zoom", 1.0)
    if z <= 1.0:
        return r
    b = r["boxes"][r["scores"] >= 0.25] * r["scale"]
    med = float(np.median(np.minimum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]))) if len(b) >= 3 else 0.0
    big = max(img.shape[:2]) * r["scale"] * z
    if med >= cfg.get("zoom_below", 32) or big > cfg.get("zoom_max_side", 8000):
        return r
    r2 = detect_raw(model, img, cfg, round(r["scale"] * z, 3), tag)
    out = dict(r)
    for k in ("boxes", "scores", "cls", "cut", "src"):
        out[k] = np.concatenate([r[k], r2[k]])
    out.update(ms=r["ms"] + r2["ms"], n_tiles=r["n_tiles"] + r2["n_tiles"], cached=r["cached"] and r2["cached"],
               zoomed=True)
    return out


def run_board(M, img, cfg, p, stages=("any", "fine", "tree")):
    """M: dict(any=YOLO, fine=YOLO, rf=forest, tags=..., fine_names=[...], to_name=fn).
    Returns per-stage results with timings in ms (detection timings of cached results are the ones measured
    when the model actually ran)."""
    R, scale = {}, None
    if M.get("any") is not None:
        raw = detect_raw(M["any"], img, cfg, None, M["tags"]["any"])
        scale = raw["scale"]
        raw = detect_multi(M["any"], img, cfg, scale, M["tags"]["any"])
        with Clock() as c:
            B, S, _ = postprocess(raw, p["conf_any"], p, agnostic=True)
        R["any"] = dict(boxes=B, scores=S, labels=["component"] * len(B), t_det=raw["ms"] + raw["probe_ms"],
                        t_post=c.ms, cached=raw["cached"], scale=raw["scale"], n_tiles=raw["n_tiles"], n_raw=int((raw["scores"] >= p["conf_any"]).sum()),
                        zoomed=raw.get("zoomed", False))
    if "fine" in stages and M.get("fine") is not None:
        raw = detect_raw(M["fine"], img, cfg, scale, M["tags"]["fine"]) if scale is None else             detect_multi(M["fine"], img, cfg, scale, M["tags"]["fine"])
        with Clock() as c:
            B, S, C = postprocess(raw, p["conf_fine"], p, agnostic=False)
        R["fine"] = dict(boxes=B, scores=S, labels=[M["fine_names"][int(k)] for k in C], cls=C,
                         t_det=raw["ms"] + raw["probe_ms"], t_post=c.ms, cached=raw["cached"], scale=raw["scale"],
                         n_tiles=raw["n_tiles"], n_raw=int((raw["scores"] >= p["conf_fine"]).sum()))
    if "any" in R and "fine" in R:
        with Clock() as c:
            fu = fuse((R["any"]["boxes"], R["any"]["scores"]), (R["fine"]["boxes"], R["fine"]["scores"], R["fine"]["cls"]),
                      M["fine_names"], p)
        fu["t_fuse"] = c.ms
        R["both"] = fu
        if "vote" in stages and M.get("rf") is not None:
            # model 2 names what it can; every part it could not name, or named with confidence < fine_min,
            # goes to the forest, which names it when its probability >= tree_min_prob
            ask = [l == "?" or c < p["fine_min"] for l, c in zip(fu["labels"], fu["label_conf"])]
            probs, tinfo = tree_probs(M, img, fu["boxes"], p, ask)
            L, C, S_ = list(fu["labels"]), list(fu["label_conf"]), list(fu["src"])
            for i, pt in enumerate(probs):
                if pt:
                    c = max(pt, key=pt.get)
                    if pt[c] >= p["tree_min_prob"]:
                        S_[i] = "tree" if L[i] != c else "fine+tree"
                        L[i], C[i] = c, pt[c]
            tinfo["n_changed"] = sum(1 for a_, b_ in zip(L, fu["labels"]) if a_ != b_)
            R["vote"] = dict(boxes=fu["boxes"], scores=fu["scores"], labels=L, label_conf=C, src=S_, tree=tinfo)
        if "tree" in stages and M.get("rf") is not None:
            fu3 = dict(boxes=fu["boxes"], scores=fu["scores"], labels=list(fu["labels"]),
                       label_conf=list(fu["label_conf"]), src=list(fu["src"]))
            fu3["tree"] = tree_refine(fu3, img, M["rf"], p, M["to_name"])
            R["all3"] = fu3
    if "tree_all" in stages and M.get("rf") is not None and "any" in R:
        a = R["any"]
        probs, tinfo = tree_probs(M, img, a["boxes"], p)
        L, C, S_ = [], [], []
        for pr in probs:
            c = max(pr, key=pr.get) if pr else None
            ok = c is not None and pr[c] >= p["tree_min_prob"]
            L.append(c if ok else "?"); C.append(pr[c] if ok else 0.0); S_.append("tree" if ok else "any")
        tinfo["n_changed"] = S_.count("tree")
        R["any_tree"] = dict(boxes=a["boxes"], scores=a["scores"], labels=L, label_conf=C, src=S_, tree=tinfo)
    return R


# ================================================================== notebook views
def show(img, width=1400):
    try:
        from IPython.display import Image, display
        display(Image(data=cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 88])[1].tobytes(), width=width))
    except ImportError:
        pass


def _ms(v):
    return f"{v / 1000:.2f} s" if v >= 1000 else f"{v:.0f} ms"


STAGES = {1: ("any",), 2: ("any", "tree_all"), 3: ("any", "fine", "tree"), 4: ("any", "fine", "vote")}
TITLE = {1: "1 - model 1 only (where)", 2: "2 - model 1 + forest (forest names every part)",
         3: "3 - model 1 + model 2, forest only for '?'", 4: "4 - all parts: model 2 names, forest for '?' and weak names"}
RESULT = {2: "any_tree", 3: "all3", 4: "vote"}


def stage_result(R, stage):
    """Final boxes of a stage + its time split."""
    a = R["any"]
    t = {"any: detect": a["t_det"], "any: merge": a["t_post"]}
    if stage == 1:
        return a["boxes"], a["scores"], a["labels"], ["any"] * len(a["boxes"]), [float(s) for s in a["scores"]], t
    if stage != 2:
        f = R["fine"]
        t.update({"fine: detect": f["t_det"], "fine: merge": f["t_post"], "fuse": R["both"]["t_fuse"]})
    r = R[RESULT[stage]]
    t["tree"] = r["tree"]["ms"]
    return r["boxes"], r["scores"], r["labels"], r["src"], r["label_conf"], t


def view(M, img, gt, cfg, p, stage, text="auto", show_errors=True, show_img=True, name=""):
    """Run (cached) + draw + print speed and accuracy of one stage. gt = (boxes, labels) or None."""
    R = run_board(M, img, cfg, p, STAGES[stage])
    B, S, L, src, lconf, t = stage_result(R, stage)
    total = sum(t.values())
    a = R["any"]
    lines = [f"time {_ms(total)}" + (" (cached run)" if a["cached"] else ""),
             f"photo scale {a['scale']:g}, {a['n_tiles']} tiles"]
    ev = None
    if gt is not None and len(gt[0]):
        GB, GL = gt
        if stage == 1:
            keep = [i for i, l in enumerate(GL) if l != "Marker"]
            GB, GL = GB[keep], [GL[i] for i in keep]
        ev = evaluate(B, S, L, GB, GL, typed=stage > 1)
        lines.append(f"found P {ev['P'] * 100:.0f}%  R {ev['R'] * 100:.0f}%  F1 {ev['F1'] * 100:.0f}%")
        if stage > 1:
            lines.append(f"type correct {ev['type_acc'] * 100:.0f}% of found")
        lines.append(f"duplicates {ev['dup']}")
    conf_txt = f"mean confidence {np.mean(lconf) * 100:.0f}%" if len(lconf) else "no element"
    lines.append(conf_txt)

    print(f"\n=== {TITLE[stage]}  {name}")
    print(f"elements: {len(B)}" + (f"   labels on the board: {len(ev['st_gt'])}" if ev else "   (no labels for this photo: accuracy = model confidence only)"))
    print("time:", "  |  ".join(f"{k} {_ms(v)}" for k, v in t.items()), f"  ||  TOTAL {_ms(total)}")
    if a["cached"]:
        print("     (the models did not run again: same photo and detection settings -> cached result; the times are from the real run)")
    if ev:
        print(f"accuracy (IoU>=0.5): precision {ev['P'] * 100:.1f}%  recall {ev['R'] * 100:.1f}%  F1 {ev['F1'] * 100:.1f}%"
              f"  | duplicates {ev['dup']}  missed {ev['st_gt'].count('miss')}")
        print(f"     'not correct' boxes: {ev['loc']} box size/position off (part is right), "
              f"{ev['unl']} with NO label near them -> look at the orange boxes on the error map: "
              f"if they are real parts, the board's labels are incomplete and the real precision is higher")
        if stage > 1:
            print(f"type: correct for {ev['type_acc'] * 100:.1f}% of the found parts  | typed F1 {ev['F1t'] * 100:.1f}%"
                  f"  | '?' (no name) {L.count('?')}")
    print(conf_txt)
    if stage > 1:
        tr = R[RESULT[stage]]["tree"]
        new = tr.get("n_new", tr["n_asked"])
        print(f"random forest: asked about {tr['n_asked']} parts ({new} new, {_ms(tr['ms'])}"
              + (f", {tr['ms'] / new:.0f} ms per part" if new else ", all from cache") + f"), changed/gave the name of {tr['n_changed']}")
        from collections import Counter
        print("names from:", dict(Counter(src)), "  (fine = model 2, fine+ = found only by model 2, tree = forest, "
              "fine+tree = both agree, any = no name '?')")
    if show_img:
        if stage == 1:
            show(draw_elements(img, B, lines))
        else:
            show(draw_types(img, B, L, lconf, src, text, lines=lines)[0])
        if ev and show_errors:
            try:
                from IPython.display import HTML, display
                display(HTML('<div style="height:48px"></div><div style="direction:rtl;text-align:right;font-family:Tahoma;'
                             'font-size:15px;border-top:2px solid #888;padding-top:10px">'
                             '<b>نقشه‌ی خطا در مقایسه با برچسب‌های این برد:</b> سبز = درست، زرد = پیدا شده ولی نوع اشتباه، '
                             'آبی روشن = جای جعبه کمی فرق دارد، نارنجی = این‌جا برچسبی نیست (اگر روی عکس المان است، '
                             'برچسب‌زن جا انداخته و خطای مدل نیست)، بنفش = تکراری، قرمز = المان برچسب‌خورده‌ای که پیدا نشد</div>'))
            except ImportError:
                pass
            show(draw_errors(img, B, ev, gt[0] if stage > 1 else GB, lines[:1]))
    return dict(R=R, boxes=B, labels=L, ev=ev, t=t, total=total)


def benchmark(M, boards, cfg, p, read_gt, progress=True):
    """Speed and accuracy over many labelled boards. boards: list of image paths; read_gt(path, W, H) -> (B, L).
    Compares the old merge ('simple' = NMS only) with the big-part fix ('smart')."""
    rows = {k: dict(tp=0, n=0, g=0, tpt=0, dup=0, q=0) for k in
            ("1 any (old NMS)", "1 any (big-part fix)", "  fine alone", "2 any+fine", "3 any+fine+forest")}
    times = {k: [] for k in ("any", "fine", "tree", "stage 1", "stage 2", "stage 3")}
    px = []
    for n, f in enumerate(boards, 1):
        img = cv2.imread(f)
        H, W = img.shape[:2]
        GB, GL = read_gt(f, W, H)
        ga = [i for i, l in enumerate(GL) if l != "Marker"]
        GBa, GLa = GB[ga], [GL[i] for i in ga]
        R = run_board(M, img, cfg, p, STAGES[3])
        old = postprocess(detect_raw(M["any"], img, cfg, R["any"]["scale"], M["tags"]["any"]), p["conf_any"], dict(p, mode="simple"), True)
        cases = {"1 any (old NMS)": (old[0], old[1], ["component"] * len(old[0]), GBa, GLa, False),
                 "1 any (big-part fix)": (R["any"]["boxes"], R["any"]["scores"], R["any"]["labels"], GBa, GLa, False),
                 "  fine alone": (R["fine"]["boxes"], R["fine"]["scores"], R["fine"]["labels"], GB, GL, True),
                 "2 any+fine": (R["both"]["boxes"], R["both"]["scores"], R["both"]["labels"], GB, GL, True),
                 "3 any+fine+forest": (R["all3"]["boxes"], R["all3"]["scores"], R["all3"]["labels"], GB, GL, True)}
        for k, (B, S, L, gb, gl, typed) in cases.items():
            ev = evaluate(B, S, L, gb, gl, typed=typed)
            r = rows[k]
            r["tp"] += ev["tp"]; r["n"] += ev["n_pred"]; r["g"] += ev["n_gt"]; r["dup"] += ev["dup"]
            r["tpt"] += round(ev["type_acc"] * ev["tp"]); r["q"] += list(L).count("?")
        ta = R["any"]["t_det"] + R["any"]["t_post"]
        tf_ = R["fine"]["t_det"] + R["fine"]["t_post"] + R["both"]["t_fuse"]
        tt = R["all3"]["tree"]["ms"]
        for k, v in (("any", ta), ("fine", tf_), ("tree", tt), ("stage 1", ta), ("stage 2", ta + tf_), ("stage 3", ta + tf_ + tt)):
            times[k].append(v)
        px.append(W * H / 1e6)
        if progress:
            print(f"\r{n}/{len(boards)}  {os.path.basename(f)[:40]:40s}", end="", flush=True)
    print()
    f1 = lambda a, b: 2 * a * b / max(a + b, 1e-9)
    print(f"{'':24s}{'found P':>9}{'R':>7}{'F1':>7}{'type ok':>9}{'typed F1':>10}{'dups':>7}{'no name':>9}")
    for k, r in rows.items():
        P, Rr = r["tp"] / max(r["n"], 1), r["tp"] / max(r["g"], 1)
        typed = not k.startswith("1")
        Pt, Rt = r["tpt"] / max(r["n"], 1), r["tpt"] / max(r["g"], 1)
        print(f"{k:24s}{P * 100:8.1f}%{Rr * 100:6.1f}%{f1(P, Rr) * 100:6.1f}%"
              + (f"{r['tpt'] / max(r['tp'], 1) * 100:8.1f}%{f1(Pt, Rt) * 100:9.1f}%" if typed else f"{'-':>9}{'-':>10}")
              + f"{r['dup']:7d}{r['q'] if typed else 0:9d}")
    print(f"\nspeed per board (mean of {len(boards)} boards, {np.mean(px):.1f} megapixel on average; times of the real model runs):")
    for k, v in times.items():
        print(f"  {k:10s} mean {_ms(np.mean(v)):>9}   median {_ms(np.median(v)):>9}   max {_ms(np.max(v)):>9}")
    return rows, times


def final_view(M, img, cfg, p, name="", out_dir=None, text="auto", gt=None):
    """The quickest complete answer: one picture with every part and its type (any + fine, the forest only for
    the parts the fine model could not name), plus a CSV of all parts."""
    R = run_board(M, img, cfg, p, STAGES[4] if M.get("rf") is not None else ("any", "fine"))
    st = 4 if "vote" in R else None
    if st is None:   # no forest: model 1 + model 2 only
        R["vote"] = dict(R["both"], tree=dict(n_asked=0, n_changed=0, ms=0.0))
        st = 4
    B, S, L, src, lconf, t = stage_result(R, st)
    total = sum(t.values())
    from collections import Counter
    n_tree = src.count("tree")
    lines = [f"{len(B)} parts in {_ms(total)}" + (" (cached run)" if R["any"]["cached"] else ""),
             f"mean confidence {np.mean(lconf) * 100:.0f}%" if len(lconf) else "no part found",
             f"named by forest: {n_tree}, unknown: {L.count('?')}"]
    vis, counts = draw_types(img, B, L, lconf, src, text, lines=lines)
    show(vis)
    print(f"{name}: {len(B)} parts | time {_ms(total)}  (" + ", ".join(f"{k} {_ms(v)}" for k, v in t.items()) + ")")
    print("per type:", dict(sorted(counts.items(), key=lambda kv: -kv[1])))
    if R["any"].get("zoomed"):
        print("tiny parts: the photo was also checked enlarged")
    if gt is not None and len(gt[0]):
        ev = evaluate(B, S, L, gt[0], gt[1], typed=True)
        print(f"labels of this board: found {ev['tp']} of {ev['n_gt']} parts ({ev['R'] * 100:.0f}%), "
              f"type right for {ev['type_acc'] * 100:.0f}% of them")
    if out_dir:
        import csv
        os.makedirs(out_dir, exist_ok=True)
        stem = os.path.splitext(name or "board")[0]
        cv2.imwrite(os.path.join(out_dir, f"{stem}_result.jpg"), vis, [cv2.IMWRITE_JPEG_QUALITY, 92])
        with open(os.path.join(out_dir, f"{stem}_parts.csv"), "w", newline="", encoding="utf8") as fh:
            w = csv.writer(fh); w.writerow(["id", "type", "type_conf", "found_conf", "named_by", "x0", "y0", "x1", "y1"])
            for i in range(len(B)):
                w.writerow([i + 1, L[i], f"{lconf[i]:.3f}", f"{S[i]:.3f}", src[i], *[int(v) for v in B[i]]])
        print("saved:", os.path.join(out_dir, f"{stem}_result.jpg"), "+ _parts.csv")
    return R


def settings_for(cfg, p, mode, conf):
    """Per-cell settings. mode 'complete' = do not miss parts: more tile overlap, an extra enlarged pass on boards
    with tiny parts, and parts that only the 'fine' model saw are added at a lower confidence."""
    c, q = dict(cfg), dict(p)
    q["conf_any"] = q["conf_fine"] = float(conf)
    if mode == "complete":
        c.update(overlap=max(c["overlap"], 0.35), zoom=1.5, zoom_below=32)
        q["add_fine_conf"] = min(q["add_fine_conf"], 0.25)
    return c, q

In [ ]:
%%writefile /content/work/tree_features.py
"""Hand-made features of the component classifier (random forest) - copied unchanged from
درخت/درخت/final_33.ipynb (the notebook that reached 92.6 % on 1195 crops), plus a small wrapper.

extract_features_advanced() accepts a file path OR a BGR image (a crop cut from a board photo).
"""
import os
import warnings

import cv2
import numpy as np

warnings.filterwarnings("ignore")   # tiny crops give empty regions -> 'Mean of empty slice' etc. (harmless, value 0)
np.seterr(all="ignore")

IMAGE_SIZE = 256


def resize_with_aspect_ratio(image, size=IMAGE_SIZE):
    h, w = image.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h*scale), int(w*scale)
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    y0 = (size - nh)//2
    x0 = (size - nw)//2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

COLOR_RANGES = {
    'yellow': ((20, 100, 100), (35, 255, 255)),
    'cream':  ((15, 20, 180),  (40, 120, 255)),
    'brown':  ((5, 80, 20),    (20, 255, 180)),
    'orange': ((5, 120, 120),  (30, 255, 255)),
    'black':  ((0, 0, 0),      (180, 255, 50)),
    'gray':   ((0, 0, 50),     (180, 40, 200)),
    'silver': ((0, 0, 200),    (180, 50, 255)),
    'green':  ((40, 50, 50),   (80, 255, 255)),
    'red1':   ((0, 100, 100),  (10, 255, 255)),
    'red2':   ((170, 100, 100),(180, 255, 255)),
}

# قبل از توابع، COLOR_MAP را آپدیت کن:
COLOR_MAP = {
    'black': 0, 'brown': 1, 'cream': 2, 'gray': 3, 'silver': 4,
    'yellow': 5, 'orange': 6, 'green': 7, 'red': 8,
    # اضافه کردن رنگ‌های جدید
    'blue': 9, 'white': 10, 'unknown': -1
}



def classify_color(image):
    """
    تشخیص رنگ با دقت بالاتر - بهبود یافته
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    
    # توسعه محدوده رنگ‌ها برای دقت بیشتر
    EXPANDED_COLOR_RANGES = {
        'yellow': [
            ((20, 100, 100), (35, 255, 255)),  # زرد استاندارد
            ((15, 150, 150), (30, 255, 255)),  # زرد روشن
            ((25, 80, 120), (40, 255, 255)),   # زرد متمایل به نارنجی
        ],
        'cream': [
            ((15, 20, 180), (25, 100, 255)),   # کرم روشن
            ((20, 10, 190), (30, 80, 255)),    # کرم
            ((10, 15, 170), (35, 90, 240)),    # کرم تیره
        ],
        'brown': [
            ((5, 80, 20), (20, 255, 180)),     # قهوه‌ای روشن
            ((0, 100, 30), (20, 255, 150)),    # قهوه‌ای
            ((5, 120, 40), (25, 255, 120)),    # قهوه‌ای تیره
        ],
        'orange': [
            ((5, 120, 120), (25, 255, 255)),   # نارنجی پررنگ
            ((10, 100, 150), (30, 255, 255)),  # نارنجی
            ((0, 100, 130), (20, 255, 230)),   # نارنجی روشن
        ],
        'black': [
            ((0, 0, 0), (180, 255, 50)),       # سیاه استاندارد
            ((0, 0, 0), (180, 100, 30)),       # سیاه خالص
            ((0, 0, 0), (180, 150, 40)),       # سیاه متمایل به خاکستری
        ],
        'gray': [
            ((0, 0, 50), (180, 50, 200)),      # خاکستری روشن
            ((0, 0, 30), (180, 30, 150)),      # خاکستری
            ((0, 0, 70), (180, 40, 180)),      # خاکستری تیره
        ],
        'silver': [
            ((0, 0, 180), (180, 50, 255)),     # نقره‌ای
            ((0, 0, 150), (180, 30, 230)),     # نقره‌ای روشن
            ((0, 0, 200), (180, 40, 255)),     # نقره‌ای درخشان
        ],
        'green': [
            ((40, 50, 50), (80, 255, 255)),    # سبز الکترونیکی
            ((36, 25, 25), (70, 255, 200)),    # سبز روشن
            ((45, 100, 50), (85, 255, 180)),   # سبز تیره
        ],
        'red': [  # ترکیب دو محدوده قرمز
            ((0, 100, 100), (10, 255, 255)),   # قرمز روشن
            ((170, 100, 100), (180, 255, 255)), # قرمز تیره
        ],
        'blue': [  # اضافه کردن آبی برای قطعات الکترونیکی
            ((90, 50, 50), (130, 255, 255)),   # آبی
            ((100, 60, 60), (140, 255, 200)),  # آبی روشن
        ],
        'white': [  # اضافه کردن سفید
            ((0, 0, 200), (180, 30, 255)),     # سفید
        ]
    }
    
    # مناطق نمونه‌گیری استراتژیک
    height, width = IMAGE_SIZE, IMAGE_SIZE
    sample_regions = [
        # مرکز (مهمترین منطقه)
        hsv[height//2-30:height//2+30, width//2-30:width//2+30],
        # چهار گوشه
        hsv[:height//3, :width//3],
        hsv[:height//3, 2*width//3:],
        hsv[2*height//3:, :width//3],
        hsv[2*height//3:, 2*width//3:],
        # لبه‌ها (برای تشخیص الکترود)
        hsv[:, :width//6],
        hsv[:, 5*width//6:],
    ]
    
    region_weights = [1.5, 0.5, 0.5, 0.5, 0.5, 0.7, 0.7]  # وزن مناطق
    
    color_votes = {}
    color_scores = {}
    
    for idx, region in enumerate(sample_regions):
        if region.size == 0:
            continue
            
        weight = region_weights[idx]
        
        # نمونه‌گیری تصادفی از پیکسل‌ها در منطقه
        if region.size > 100:
            flat_region = region.reshape(-1, 3)
            num_samples = min(50, len(flat_region))
            indices = np.random.choice(len(flat_region), num_samples, replace=False)
            samples = flat_region[indices]
        else:
            samples = region.reshape(-1, 3)
        
        for h, s, v in samples:
            # فیلتر پیکسل‌های غیرمعمول
            if v < 20 or v > 240:  # خیلی تیره یا خیلی روشن
                continue
                
            # تشخیص رنگ بر اساس HSV
            matched = False
            for color_name, ranges in EXPANDED_COLOR_RANGES.items():
                for (lower, upper) in ranges:
                    if (lower[0] <= h <= upper[0]) and \
                       (lower[1] <= s <= upper[1]) and \
                       (lower[2] <= v <= upper[2]):
                        
                        # محاسبه امتیاز بر اساس فاصله از مرکز محدوده
                        h_center = (lower[0] + upper[0]) / 2
                        s_center = (lower[1] + upper[1]) / 2
                        v_center = (lower[2] + upper[2]) / 2
                        
                        h_distance = 1 - min(abs(h - h_center) / 90, 1)
                        s_distance = 1 - min(abs(s - s_center) / 255, 1)
                        v_distance = 1 - min(abs(v - v_center) / 255, 1)
                        
                        score = (h_distance * 0.4 + s_distance * 0.3 + v_distance * 0.3) * weight
                        
                        # رأی برای رنگ
                        color_votes[color_name] = color_votes.get(color_name, 0) + 1
                        color_scores[color_name] = color_scores.get(color_name, 0) + score
                        matched = True
                        break
                if matched:
                    break
            
            # اگر رنگی پیدا نشد و اشباع کم است، احتمالاً خاکستری/سفید/سیاه
            if not matched and s < 40:
                if v > 200:
                    color_name = 'white'
                elif v > 100:
                    color_name = 'gray'
                else:
                    color_name = 'black'
                
                color_votes[color_name] = color_votes.get(color_name, 0) + 1
                color_scores[color_name] = color_scores.get(color_name, 0) + weight
    
    # اگر هیچ رنگی پیدا نشد
    if not color_scores:
        # تحلیل بر اساس میانگین HSV
        mean_hsv = np.mean(hsv.reshape(-1, 3), axis=0)
        h, s, v = mean_hsv
        
        if s < 30:
            if v > 180:
                return 'white'
            elif v > 80:
                return 'gray'
            else:
                return 'black'
        elif 15 <= h <= 35:
            return 'yellow'
        elif 5 <= h <= 25:
            if s > 80 and v > 120:
                return 'orange'
            else:
                return 'brown'
        elif 36 <= h <= 85:
            return 'green'
        elif 90 <= h <= 130:
            return 'blue'
        elif h <= 10 or h >= 170:
            return 'red'
        else:
            return 'unknown'
    
    # ترکیب رأی و امتیاز
    final_scores = {}
    for color in set(color_votes.keys()) | set(color_scores.keys()):
        votes = color_votes.get(color, 0)
        score = color_scores.get(color, 0)
        # میانگین وزنی از رأی و امتیاز
        final_scores[color] = (votes * 0.4) + (score * 0.6)
    
    # انتخاب رنگ برنده
    best_color = max(final_scores, key=final_scores.get)
    
    # اگر اختلاف امتیاز کم است، دقت بیشتری انجام دهیم
    sorted_colors = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    if len(sorted_colors) > 1:
        first_score = sorted_colors[0][1]
        second_score = sorted_colors[1][1]
        
        # اگر اختلاف کمتر از 10٪ است، تحلیل بیشتری انجام دهیم
        if first_score > 0 and (first_score - second_score) / first_score < 0.1:
            # تحلیل فضای LAB برای دقت بیشتر
            lab_center = lab[height//2-20:height//2+20, width//2-20:width//2+20]
            if lab_center.size > 0:
                L_mean = np.mean(lab_center[:, :, 0])
                a_mean = np.mean(lab_center[:, :, 1])
                b_mean = np.mean(lab_center[:, :, 2])
                
                # تشخیص سفید/خاکستری بر اساس کانال L
                if L_mean > 190:
                    best_color = 'white'
                elif L_mean > 120:
                    # بین gray و silver
                    if abs(a_mean) < 10 and abs(b_mean) < 10:
                        best_color = 'gray'
                    else:
                        best_color = 'silver'
    
    # تطبیق با رنگ‌های موجود در COLOR_MAP
    if best_color not in COLOR_MAP and best_color != 'unknown':
        # نگاشت به نزدیکترین رنگ موجود
        if best_color in ['white', 'blue']:
            best_color = 'unknown'
    
    return best_color


def detect_main_circle(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    
    gray_blur = cv2.GaussianBlur(gray, (9,9), 2)
    
    circles = cv2.HoughCircles(
        gray_blur,
        cv2.HOUGH_GRADIENT,
        dp=1.1,
        minDist=int(0.6*min(h,w)),
        param1=100,
        param2=35,
        minRadius=int(0.2*min(h,w)),
        maxRadius=int(0.48*min(h,w))
    )
    
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    circle_found = 0
    if circles is not None:
        circles = np.uint16(np.around(circles.reshape(-1, 3)))
        x, y, r = max(circles, key=lambda c: c[2])
        
        mask = np.zeros(gray.shape, dtype=np.uint8)
        cv2.circle(mask, (x, y), r, 255, -1)
        filled_ratio = cv2.countNonZero(mask) / (np.pi*r*r)
        
        if filled_ratio > 0.5 and r > min(h, w) * 0.15:
            circle_found = 1
    
    if not circle_found and contours:
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area > (h * w * 0.1):
                perimeter = cv2.arcLength(cnt, True)
                if perimeter == 0:
                    continue
                
                circularity = 4 * np.pi * area / (perimeter * perimeter)
                if circularity > 0.85:
                    circle_found = 1
                    break
    
    return circle_found


def count_legs_advanced(image):
    """
    شمارش پایه با تحلیل پروفایل روشنایی در چهار نوار کناری.
    خروجی: یک عدد صحیح (تعداد کل پایه‌ها).
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    strip_w = int(W * 0.12)
    strip_h = int(H * 0.12)

    left_strip  = gray[:, :strip_w]
    right_strip = gray[:, W-strip_w:]
    top_strip   = gray[:strip_h, :]
    bottom_strip= gray[H-strip_h:, :]

    def count_legs_in_strip(strip, axis):
        sh, sw = strip.shape
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4,4))
        strip_eq = clahe.apply(strip)
        blurred = cv2.GaussianBlur(strip_eq, (5, 5), 0)

        if axis == 0:
            profile = np.mean(blurred, axis=1).astype(np.float32)
        else:
            profile = np.mean(blurred, axis=0).astype(np.float32)

        kernel_size = max(3, len(profile) // 20)
        if kernel_size % 2 == 0:
            kernel_size += 1
        profile_smooth = cv2.GaussianBlur(
            profile.reshape(-1,1), (1, kernel_size), 0
        ).flatten()

        mean_val = np.mean(profile_smooth)
        binary = (profile_smooth > mean_val).astype(np.uint8)

        transitions = np.diff(binary.astype(np.int8))
        starts = np.where(transitions == 1)[0]
        ends   = np.where(transitions == -1)[0]

        if len(starts) == 0 or len(ends) == 0:
            return 0

        if ends[0] < starts[0]:
            ends = ends[1:]
        if len(starts) > len(ends):
            starts = starts[:len(ends)]

        min_width = len(profile) * 0.01
        max_width = len(profile) * 0.25
        std_val = np.std(profile_smooth)
        min_amplitude = std_val * 0.3

        valid_legs = []
        for s, e in zip(starts, ends):
            width = e - s
            block_mean = np.mean(profile_smooth[s:e+1])
            amplitude = block_mean - mean_val
            if min_width < width < max_width and amplitude > min_amplitude:
                valid_legs.append((s, e))

        if len(valid_legs) > 1:
            centers = [(s + e) / 2 for s, e in valid_legs]
            min_distance = len(profile) * 0.02
            filtered_legs = [valid_legs[0]]
            for i in range(1, len(valid_legs)):
                if centers[i] - centers[i-1] >= min_distance:
                    filtered_legs.append(valid_legs[i])
            valid_legs = filtered_legs

        return len(valid_legs)

    left_count   = count_legs_in_strip(left_strip, 0)
    right_count  = count_legs_in_strip(right_strip, 0)
    top_count    = count_legs_in_strip(top_strip, 1)
    bottom_count = count_legs_in_strip(bottom_strip, 1)

    # تقارن
    if left_count == 0: right_count = 0
    if right_count == 0: left_count = 0
    if top_count == 0: bottom_count = 0
    if bottom_count == 0: top_count = 0

    return left_count + right_count + top_count + bottom_count



def detect_electrodes(image):
    """
    تشخیص الکترود در چهار طرف با امتیازدهی بر اساس:
    - اختلاف روشنایی نسبت به میانگین کل
    - تراکم لبه
    - خطوط عمودی/افقی
    - یکنواختی (انحراف معیار کم)
    خروجی: 1 اگر در دو طرف مقابل الکترود تشخیص داده شود، در غیر این صورت 0.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    margin_ratio = 0.30
    left_margin   = int(W * margin_ratio)
    right_margin  = int(W * margin_ratio)
    top_margin    = int(H * margin_ratio)
    bottom_margin = int(H * margin_ratio)

    regions = {
        'left':   gray[:, :left_margin],
        'right':  gray[:, W-right_margin:],
        'top':    gray[:top_margin, :],
        'bottom': gray[H-bottom_margin:, :]
    }

    global_mean = np.mean(gray)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=20,
                            minLineLength=15, maxLineGap=5)

    def evaluate_side(region, side):
        mean_val = np.mean(region)
        diff = mean_val - global_mean

        # Edge density
        if side == 'left':
            edge_region = edges[:, :left_margin]
        elif side == 'right':
            edge_region = edges[:, W-right_margin:]
        elif side == 'top':
            edge_region = edges[:top_margin, :]
        else:  # bottom
            edge_region = edges[H-bottom_margin:, :]
        edge_density = np.sum(edge_region > 0) / edge_region.size

        # Vertical/horizontal lines
        v_lines = 0
        h_lines = 0
        if lines is not None:
            for x1, y1, x2, y2 in lines.reshape(-1, 4):
                if abs(x2 - x1) < 5 and abs(y2 - y1) > 10:  # vertical line
                    if side == 'left' and x1 < left_margin:
                        v_lines += 1
                    elif side == 'right' and x1 > W - right_margin:
                        v_lines += 1
                    elif side == 'top' and y1 < top_margin:
                        h_lines += 1
                    elif side == 'bottom' and y1 > H - bottom_margin:
                        h_lines += 1

        std_val = np.std(region)

        # Scoring
        score = 0
        if diff > 5: score += 1
        if edge_density > 0.05: score += 1
        if (side in ('left','right') and v_lines >= 1) or \
           (side in ('top','bottom') and h_lines >= 1):
            score += 1
        if std_val < 30: score += 1

        return score >= 3   # الکترود محسوب می‌شود

    left_ok   = evaluate_side(regions['left'], 'left')
    right_ok  = evaluate_side(regions['right'], 'right')
    top_ok    = evaluate_side(regions['top'], 'top')
    bottom_ok = evaluate_side(regions['bottom'], 'bottom')

    # شرط نهایی
    return 1 if (left_ok and right_ok) or (top_ok and bottom_ok) else 0   

def detect_written_advanced(gray):
    """
    تشخیص بلوک‌های متن (نه کاراکترهای جداگانه).
    خروجی:
        - تعداد بلوک‌های معتبر (int)
        - مجموع پیکسل‌های متن (int)
        - وجود متن (1 اگر حداقل یک بلوک باشد) (int)
        - چگالی متن (نسبت پیکسل‌های متن به کل تصویر) (float)
    """
    height, width = gray.shape

    # 1. بهبود کنتراست
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # 2. حذف نویز
    denoised = cv2.fastNlMeansDenoising(enhanced, None, h=10,
                                        templateWindowSize=7, searchWindowSize=21)

    # 3. آستانه‌گذاری‌های مختلف
    adaptive1 = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY_INV, 11, 2)
    adaptive2 = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                      cv2.THRESH_BINARY_INV, 15, 3)
    _, otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 4. ترکیب و مورفولوژی
    combined = cv2.bitwise_or(adaptive1, adaptive2)
    combined = cv2.bitwise_or(combined, otsu)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 5))
    morphed = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel, iterations=2)

    # 5. آنالیز اجزای متصل
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(morphed, connectivity=8)

    MIN_AREA = height * width * 0.01   # حداقل 1٪ تصویر
    MAX_AREA = height * width * 0.8    # حداکثر 80٪ تصویر
    total_text_pixels = 0
    valid_blocks = 0

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if MIN_AREA <= area <= MAX_AREA:
            valid_blocks += 1
            total_text_pixels += area

    has_text = 1 if valid_blocks > 0 else 0
    text_density = total_text_pixels / (height * width)

    return valid_blocks, total_text_pixels, has_text, text_density


def extract_shape_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return 0, 0, 0, 1.0
    
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    
    circularity = (4 * np.pi * area) / (perimeter * perimeter) if perimeter > 0 else 0
    aspect_ratio = w / h if h > 0 else 0
    
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0
    
    return circularity, aspect_ratio, solidity, area



def infer_category(filename):
    f = filename.lower()
    if 'resistor' in f: return 'resistors'
    if 'capacitor' in f: return 'capacitors'
    if 'diode' in f: return 'diodes'
    if 'inductor' in f: return 'inductors'
    if 'transistor' in f: return 'transistors'
    if 'ic' in f or 'integrated' in f: return 'ICs'
    return 'unknown'

    
def extract_features_advanced(path):

    img = cv2.imread(path) if isinstance(path, str) else path
    if img is None:
        return None
    
    img = resize_with_aspect_ratio(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    
    # ویژگی‌های اصلی با توابع بهبود یافته
    color = classify_color(img)  # تابع بهبود یافته
    is_circle = detect_main_circle(img)
    electrodes = detect_electrodes(img)
    pin_count = count_legs_advanced(img)
    three_legs = 1 if pin_count >= 3 else 0
    
    # تشخیص متن با تابع بهبود یافته
    chars, area, has_text, text_density = detect_written_advanced(gray)
    
    # ویژگی‌های شکل
    circularity, aspect_ratio, solidity, contour_area = extract_shape_features(img)
    
    # ویژگی‌های شدت نور
    mean_intensity = np.mean(gray)
    std_intensity = np.std(gray)
    
    # ویژگی‌های جدید برای بهبود دقت
    # 1. تحلیل گرادیان (لبه‌ها)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    gradient_magnitude = np.sqrt(sobelx**2 + sobely**2)
    gradient_mean = np.mean(gradient_magnitude)
    gradient_std = np.std(gradient_magnitude)
    
    # 2. تحلیل بافت ساده (واریانس محلی)
    from scipy.ndimage import uniform_filter
    gray_float = gray.astype(float)
    local_mean = uniform_filter(gray_float, size=5)
    local_sq_mean = uniform_filter(gray_float**2, size=5)
    local_variance = local_sq_mean - local_mean**2
    texture_mean = np.mean(local_variance)
    
    # 3. ویژگی‌های رنگ RGB
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    r_mean, g_mean, b_mean = np.mean(rgb_img, axis=(0, 1))
    r_std, g_std, b_std = np.std(rgb_img, axis=(0, 1))
    
    # 4. نسبت رنگ‌ها
    sum_rgb = r_mean + g_mean + b_mean + 1e-10
    color_ratio_r = r_mean / sum_rgb
    color_ratio_g = g_mean / sum_rgb
    color_ratio_b = b_mean / sum_rgb
    
    # 5. ویژگی‌های هیستوگرام
    hist = cv2.calcHist([gray], [0], None, [64], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    hist_mean = np.mean(hist)
    hist_std = np.std(hist)
    
    # 6. تراکم لبه‌ها
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / (height * width)
    
    # 7. ویژگی‌های موقعیت متن
    text_in_center = 1 if chars > 0 and text_density > 0.005 else 0
    
    # 8. تنوع رنگ در مرکز
    center_region = img[height//4:3*height//4, width//4:3*width//4]
    if center_region.size > 0:
        center_hsv = cv2.cvtColor(center_region, cv2.COLOR_BGR2HSV)
        center_saturation = np.mean(center_hsv[:, :, 1])
        center_value = np.mean(center_hsv[:, :, 2])
    else:
        center_saturation = 0
        center_value = 0
    
    return {
        'filename': os.path.basename(path) if isinstance(path, str) else 'crop',
        'Category': infer_category(path) if isinstance(path, str) else 'unknown',
        'color_name': color,
        'color_code': COLOR_MAP.get(color, -1),
        'is_circle': is_circle,
        'electrodes_both_ends': electrodes,
        'pin_count': pin_count,
        'three_legs': three_legs,
        'written_chars': chars,
        'written_area_sum': area,
        'has_text': has_text,
        'text_density': text_density,
        'circularity': circularity,
        'aspect_ratio': aspect_ratio,
        'solidity': solidity,
        'contour_area': contour_area,
        'mean_intensity': mean_intensity,
        'std_intensity': std_intensity,
        
        # ویژگی‌های جدید (اضافه شده)
        'gradient_mean': float(gradient_mean),
        'gradient_std': float(gradient_std),
        'texture_mean': float(texture_mean),
        'r_mean': float(r_mean),
        'g_mean': float(g_mean),
        'b_mean': float(b_mean),
        'r_std': float(r_std),
        'g_std': float(g_std),
        'b_std': float(b_std),
        'color_ratio_r': float(color_ratio_r),
        'color_ratio_g': float(color_ratio_g),
        'color_ratio_b': float(color_ratio_b),
        'hist_mean': float(hist_mean),
        'hist_std': float(hist_std),
        'edge_density': float(edge_density),
        'text_in_center': text_in_center,
        'center_saturation': float(center_saturation),
        'center_value': float(center_value),
    }


# ------------------------------------------------------------------ wrapper used by the test notebook
FEATURE_COLS = [
    'color_code', 'is_circle', 'electrodes_both_ends', 'three_legs',
    'pin_count', 'written_chars', 'written_area_sum', 'has_text',
    'text_density', 'circularity', 'aspect_ratio', 'solidity',
    'contour_area', 'mean_intensity', 'std_intensity',
    'gradient_mean', 'gradient_std', 'texture_mean',
    'r_mean', 'g_mean', 'b_mean', 'r_std', 'g_std', 'b_std',
    'color_ratio_r', 'color_ratio_g', 'color_ratio_b',
    'hist_mean', 'hist_std', 'edge_density',
    'text_in_center', 'center_saturation', 'center_value',
]
# tree class -> name used by the YOLO 'fine' model
TO_YOLO = {'resistors': 'Resistor', 'capacitors': 'Capacitor', 'ICs': 'IC', 'diodes': 'Diode',
           'inductors': 'Inductor', 'transistors': 'Transistor'}


def features_row(img):
    """Feature vector of one BGR crop (same order as FEATURE_COLS). classify_color samples pixels at random:
    the seed makes the answer repeatable."""
    np.random.seed(0)
    f = extract_features_advanced(img)
    return [float(f[c]) for c in FEATURE_COLS]


def features_many(crops, workers=None):
    """Features of many crops in parallel threads (OpenCV releases the GIL)."""
    from concurrent.futures import ThreadPoolExecutor
    if not crops:
        return np.zeros((0, len(FEATURE_COLS)))
    with ThreadPoolExecutor(workers or min(8, os.cpu_count() or 2)) as ex:
        return np.array(list(ex.map(features_row, crops)))


def train_forest(csv_path):
    """Same model as final_33.ipynb: RandomForest(100 trees, depth 10), trained on the saved feature table."""
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    df = pd.read_csv(csv_path)
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(df[FEATURE_COLS].fillna(0).values, df['Category'].values)
    return rf


def load_forest(joblib_path, csv_path=None):
    """Load the saved forest; if this scikit-learn cannot read it, rebuild it from the feature table (a few seconds)."""
    import warnings
    try:
        import joblib
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            rf = joblib.load(joblib_path)
        rf.predict(np.zeros((1, len(FEATURE_COLS))))
        return rf, 'loaded'
    except Exception as e:
        if csv_path and os.path.exists(csv_path):
            return train_forest(csv_path), f'rebuilt from {os.path.basename(csv_path)} ({type(e).__name__})'
        raise


def classify(rf, crops, workers=None):
    """crops: list of BGR images -> list of (tree_class, probability)."""
    X = features_many(crops, workers)
    if not len(X):
        return []
    P = rf.predict_proba(X)
    return [(rf.classes_[i], float(p[i])) for p, i in zip(P, P.argmax(1))]

<div style="direction:rtl;text-align:right;font-family:B Lotus, B Nazanin, Tahoma">

## اجرا

</div>

In [ ]:
#@title 1) آماده‌سازی (یک بار): Drive، کتابخانه، بارگذاری مدل‌ها، بردهای تست
import os, sys, glob, random, zipfile, subprocess, importlib
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Colab نیست - RUNS_DIR را دستی بنویس")

def drive_id(path):
    try:
        return os.getxattr(path, "user.drive.id").decode()
    except Exception:
        return None

def find_runs():
    if RUNS_DIR:
        return RUNS_DIR
    pats = [f"/content/drive/.shortcut-targets-by-id/{RUNS_FOLDER_ID}/{RUNS_NAME}",
            f"/content/drive/MyDrive/{RUNS_NAME}", f"/content/drive/MyDrive/*/{RUNS_NAME}", f"/content/drive/MyDrive/*/*/{RUNS_NAME}",
            f"/content/drive/.shortcut-targets-by-id/*/{RUNS_NAME}", f"/content/drive/.shortcut-targets-by-id/*/*/{RUNS_NAME}"]
    cands = [c for pat in pats for c in glob.glob(pat) if os.path.isdir(c)]
    for c in cands:
        if RUNS_FOLDER_ID in c or drive_id(c) == RUNS_FOLDER_ID:
            return c
    with_w = [c for c in cands if os.path.exists(f"{c}/final/any_best.pt")]
    if with_w:
        return with_w[0]
    raise FileNotFoundError(f"پوشه‌ی {RUNS_NAME} دیده نمی‌شود: https://drive.google.com/drive/folders/{RUNS_FOLDER_ID} "
                            "▸ راست‌کلیک ▸ Organize ▸ Add shortcut ▸ My Drive، بعد دوباره اجرا کن")

RUNS = find_runs()
print("پوشه:", RUNS)

# same Ultralytics version as the training (the weights were saved with it)
try:
    from importlib.metadata import version
    _v = version("ultralytics")
except Exception:
    _v = None
if _v != "8.4.174":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.174"], check=True)
import warnings
warnings.filterwarnings("ignore")
import ultralytics, torch, cv2, numpy as np
np.seterr(all="ignore")
import pcb_infer as pi, tree_features as tf
importlib.reload(tf); importlib.reload(pi)
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ندارد (CPU)")

def find_weights(stage):
    """final/<stage>_best.pt (training finished) or else the best.pt of the run that is still going"""
    for w in (f"{RUNS}/final/{stage}_best.pt", *sorted(glob.glob(f"{RUNS}/{stage}_*/weights/best.pt"))):
        if os.path.exists(w):
            return w
    raise FileNotFoundError(f"وزن مدل {stage} در {RUNS} پیدا نشد (final/{stage}_best.pt یا {stage}_yolo26s/weights/best.pt)")
ANY_W, FINE_W = find_weights("any"), find_weights("fine")
M = dict(tags=dict(any=pi.model_tag(ANY_W) + str(DEV), fine=pi.model_tag(FINE_W) + str(DEV)))
M["any"] = pi.load_yolo(ANY_W, DEV, IMGSZ, half=DEV != "cpu")
M["fine"] = pi.load_yolo(FINE_W, DEV, IMGSZ, half=DEV != "cpu")
M["fine_names"] = list(M["fine"].names.values())
M["to_name"] = lambda c: tf.TO_YOLO.get(c, c)
print(f"any  : {ANY_W}  (warm-up {M['any'].warmup_ms:.0f} ms)")
print(f"fine : {FINE_W}  (warm-up {M['fine'].warmup_ms:.0f} ms)  classes {M['fine_names']}")

_tree = {"boards": ("tree_rf_boards.joblib", "tree_features_boards.csv"), "original": ("tree_rf.joblib", "tree_features.csv")}[TREE_MODEL]
_tj, _tc = f"{RUNS}/tree/{_tree[0]}", f"{RUNS}/tree/{_tree[1]}"
if os.path.exists(_tj) or os.path.exists(_tc):
    M["rf"], _how = tf.load_forest(_tj, _tc)
    print(f"tree : {_tj}  ({_how})  classes {list(M['rf'].classes_)}")
else:
    M["rf"] = None
    print(f"⚠️ مدل درخت پیدا نشد ({_tj}) - مرحله‌ی ۳ بدون درخت است. پوشه‌ی tree را در {RUNS} بگذار.")

# labelled test boards (46 boards never used in training) - only these files are read from the zip
BT = f"{WORK}/final_dataset_hr/boards_test"
if not glob.glob(f"{BT}/images/*.jpg"):
    zp = f"{RUNS}/pcb_data_hr.zip"
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if n.startswith("final_dataset_hr/boards_test/"):
                    z.extract(n, WORK)
    else:
        print("⚠️ pcb_data_hr.zip پیدا نشد: فقط عکس خودت را می‌توانی تست کنی")
BOARDS = sorted(glob.glob(f"{BT}/images/*.jpg"))
GT_NAMES = M["fine_names"]

def read_gt(path, W, H):
    t = os.path.join(os.path.dirname(os.path.dirname(path)), "labels_fine", os.path.splitext(os.path.basename(path))[0] + ".txt")
    return pi.read_labels(t, W, H, GT_NAMES)

print(f"\nبردهای تست: {len(BOARDS)}")
for i, b in enumerate(BOARDS, 1):
    print(f"{i:3d} {os.path.basename(b)[7:-8][:30]:30s}", end="\n" if i % 4 == 0 else "")
print()

In [ ]:
#@title 2) انتخاب یک عکس (فوری؛ فقط همین عکس بررسی می‌شود)
#@markdown شماره‌ی برد تست از لیست بلوک ۱؛ **0 = تصادفی**
SAMPLE = 0  #@param {type:"integer"}
#@markdown یا عکس خودت: مسیر روی Drive، یا کلمه‌ی `upload` برای دکمه‌ی آپلود (خالی = برد تست)
MY_IMAGE = ""  #@param {type:"string"}
#@markdown اختیاری: فایل برچسب YOLO همان عکس (برای درصد دقت واقعی)
MY_LABELS = ""  #@param {type:"string"}

if MY_IMAGE.strip() == "upload":
    from google.colab import files
    _up = files.upload()
    IMG_PATH = os.path.join(WORK, list(_up)[0]); open(IMG_PATH, "wb").write(list(_up.values())[0])
elif MY_IMAGE.strip():
    IMG_PATH = MY_IMAGE.strip()
else:
    assert BOARDS, "برد تستی نیست - عکس خودت را بده"
    k = SAMPLE if 1 <= SAMPLE <= len(BOARDS) else random.randint(1, len(BOARDS))
    IMG_PATH = BOARDS[k - 1]
    print(f"برد شماره {k}")
IMG = cv2.imread(IMG_PATH)
assert IMG is not None, f"عکس خوانده نشد: {IMG_PATH}"
_H, _W = IMG.shape[:2]
if MY_IMAGE.strip():
    GT = pi.read_labels(MY_LABELS.strip(), _W, _H, GT_NAMES) if MY_LABELS.strip() else None
else:
    GT = read_gt(IMG_PATH, _W, _H)
NAME = os.path.basename(IMG_PATH)
print(f"{NAME}  {_W}x{_H}  | برچسب: {len(GT[0]) if GT else 'ندارد'}")
pi.show(IMG, 700)
print("حالا بلوک ۳، ۴ یا ۵ را اجرا کن (فقط همین یک عکس بررسی می‌شود).")

In [ ]:
#@title 3) بلوک ۱: فقط مدل اول (جای المان‌ها)
#@markdown **چه می‌بینی:** عکس ۱ = همه‌ی المان‌ها با رنگ سبز. عکس ۲ (روی بردهای تست) = نقشه‌ی خطا در مقایسه با برچسب‌ها. زیر آن زمان و درصد دقت (P = چند درصد جعبه‌ها درست‌اند، R = چند درصد المان‌ها پیدا شدند).
#@markdown **حالت:** «کامل» = هیچ المانی جا نماند (همپوشانی بیشتر برش‌ها + یک نگاه بزرگ‌شده روی بردهایی که المان ریز دارند؛ کمی کندتر) | «سریع» = فقط یک نگاه
حالت = "کامل (همه‌ی المان‌ها)"  #@param ["کامل (همه‌ی المان‌ها)", "سریع"]
#@markdown **حداقل اطمینان پیدا کردن المان:** کمتر = المان بیشتر (و چند جعبه‌ی اشتباه بیشتر)
حداقل_اطمینان = 0.15  #@param {type:"slider", min:0.05, max:0.9, step:0.05}
_c, _p = pi.settings_for(CFG, P, "complete" if حالت.startswith("کامل") else "fast", حداقل_اطمینان)

OUT1 = pi.view(M, IMG, GT, _c, _p, 1, LABEL_TEXT, SHOW_ERRORS, name=NAME)

In [ ]:
#@title 4) بلوک ۲: مدل اول + درخت تصمیم (درخت نوع همه‌ی المان‌ها را می‌گوید)
#@markdown **چه می‌بینی:** مدل اول المان‌ها را پیدا می‌کند و درخت تصمیم نوع **همه**‌ی آن‌ها را می‌گوید. عکس ۱ = نوع‌ها، عکس ۲ = نقشه‌ی خطا. اگر درخت مطمئن نباشد «?». اولین اجرا کندتر است (هر المان حدود ۵۰ میلی‌ثانیه)، بعد از کش.
#@markdown **حالت:** «کامل» = هیچ المانی جا نماند (همپوشانی بیشتر برش‌ها + یک نگاه بزرگ‌شده روی بردهایی که المان ریز دارند؛ کمی کندتر) | «سریع» = فقط یک نگاه
حالت = "کامل (همه‌ی المان‌ها)"  #@param ["کامل (همه‌ی المان‌ها)", "سریع"]
#@markdown **حداقل اطمینان پیدا کردن المان:** کمتر = المان بیشتر (و چند جعبه‌ی اشتباه بیشتر)
حداقل_اطمینان = 0.1  #@param {type:"slider", min:0.05, max:0.9, step:0.05}
_c, _p = pi.settings_for(CFG, P, "complete" if حالت.startswith("کامل") else "fast", حداقل_اطمینان)

#@markdown درخت فقط وقتی نوع می‌دهد که احتمالش حداقل این باشد
حداقل_اطمینان_درخت = 0.15  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
_p["tree_min_prob"] = حداقل_اطمینان_درخت
assert M["rf"] is not None, "مدل درخت بارگذاری نشده (خروجی بلوک ۱ آماده‌سازی را ببین)"
OUT2 = pi.view(M, IMG, GT, _c, _p, 2, LABEL_TEXT, SHOW_ERRORS, name=NAME)

In [ ]:
#@title 5) بلوک ۳ (اصلی): همه‌ی المان‌ها — مدل اول پیدا می‌کند، مدل دوم نام می‌دهد، درخت برای بقیه
#@markdown **مراحل:** ۱) مدل اول (و مدل دوم) همه‌ی المان‌ها را پیدا می‌کنند ۲) مدل دوم نوع را می‌گوید ۳) هر المانی که مدل دوم نامش را نداد **یا** اطمینانش کمتر از «مدل_دوم_کمتر_از» بود به درخت داده می‌شود و اگر درخت حداقل «حداقل_اطمینان_درخت» مطمئن بود نوعش را درخت تعیین می‌کند.
#@markdown **چه می‌بینی:** عکس ۱ = همه‌ی المان‌ها با نوعشان (نام‌های درخت گوشه‌ی سفید و «(tree)» دارند). عکس ۲ (با فاصله) = نقشه‌ی خطا. زیرش زمان هر مدل و درصد دقت.
#@markdown **حالت:** «کامل» = هیچ المانی جا نماند (همپوشانی بیشتر برش‌ها + یک نگاه بزرگ‌شده روی بردهایی که المان ریز دارند؛ کمی کندتر) | «سریع» = فقط یک نگاه
حالت = "کامل (همه‌ی المان‌ها)"  #@param ["کامل (همه‌ی المان‌ها)", "سریع"]
#@markdown **حداقل اطمینان پیدا کردن المان:** کمتر = المان بیشتر (و چند جعبه‌ی اشتباه بیشتر)
حداقل_اطمینان = 0.1  #@param {type:"slider", min:0.05, max:0.9, step:0.05}
_c, _p = pi.settings_for(CFG, P, "complete" if حالت.startswith("کامل") else "fast", حداقل_اطمینان)

#@markdown حداقل اطمینان مدل دوم برای اینکه نوعی بدهد
حداقل_اطمینان_مدل_دوم = 0.1  #@param {type:"slider", min:0.05, max:0.9, step:0.05}
#@markdown اگر اطمینان مدل دوم کمتر از این بود، از درخت پرسیده می‌شود
مدل_دوم_کمتر_از = 0.3  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown درخت فقط وقتی نوع را عوض می‌کند که احتمالش حداقل این باشد
حداقل_اطمینان_درخت = 0.15  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
ذخیره_در_درایو = True  #@param {type:"boolean"}
_p.update(conf_fine=حداقل_اطمینان_مدل_دوم, fine_min=مدل_دوم_کمتر_از, tree_min_prob=حداقل_اطمینان_درخت)
assert M["rf"] is not None, "مدل درخت بارگذاری نشده (خروجی بلوک ۱ آماده‌سازی را ببین)"
OUT4 = pi.view(M, IMG, GT, _c, _p, 4, LABEL_TEXT, SHOW_ERRORS, name=NAME)
if ذخیره_در_درایو:
    import csv
    _d = f"{RUNS}/test_results"; os.makedirs(_d, exist_ok=True)
    with open(f"{_d}/{os.path.splitext(NAME)[0]}_parts.csv", "w", newline="", encoding="utf8") as fh:
        _w = csv.writer(fh); _w.writerow(["id", "type", "conf", "named_by", "x0", "y0", "x1", "y1"])
        _r = OUT4["R"]["vote"]
        for i, b in enumerate(_r["boxes"]):
            _w.writerow([i + 1, _r["labels"][i], f"{_r['label_conf'][i]:.3f}", _r["src"][i], *[int(v) for v in b]])
    print("فهرست المان‌ها ذخیره شد:", f"{_d}/{os.path.splitext(NAME)[0]}_parts.csv")